In [18]:
# --- CELL 1: CONFIG ---
BASE_MODEL = "microsoft/deberta-v3-large"
SEED = 42
EPOCHS_FROZEN = 1
EPOCHS_FINETUNE = 1
BATCH_BUNDLES = 1          # Reduced to 1 to fit safely in T4 VRAM
ACCUM_STEPS = 8            # Increased to 8 to keep the effective batch size identical
MAX_LEN = 512
HEAD_LR = 5e-4
BASE_LR = 1e-5
CONSISTENCY_WEIGHT = 1.0
SMOKE_BUNDLES = 50
OUT_NAME = "myna-v5-prototype"
print("BASE:", BASE_MODEL, "| epochs:", EPOCHS_FROZEN, "frozen +", EPOCHS_FINETUNE, "fine-tune")

BASE: microsoft/deberta-v3-large | epochs: 1 frozen + 1 fine-tune


In [3]:
# --- CELL 2: GPU + Drive + AMP Dtype Setup ---
import os, gc, torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    major, _ = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if (major >= 8 and torch.cuda.is_bf16_supported()) else torch.float16
    print("Using AMP_DTYPE:", AMP_DTYPE)
else:
    AMP_DTYPE = torch.float32
    print("STOP: no GPU. Runtime > Change runtime type > T4 GPU, then re-run.")

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = f"/content/drive/MyDrive/{OUT_NAME}"
except Exception as e:
    print("Drive fallback to /content:", e)
    OUT_DIR = f"/content/{OUT_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

PyTorch: 2.11.0+cu130 | CUDA available: True
GPU: Tesla T4
Using AMP_DTYPE: torch.float16
Mounted at /content/drive
OUT_DIR: /content/drive/MyDrive/myna-v5-prototype


In [4]:
# --- CELL 3: Clone Myna + Install ---
import os
ROOT_DIR = '/content' if os.path.isdir('/content') else os.getcwd()
REPO_DIR = os.path.join(ROOT_DIR, 'myna-repo')
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    !git pull --ff-only 2>&1 | tail -2
else:
    os.chdir(ROOT_DIR)
    !git clone -q https://github.com/aashish254/myna.git {REPO_DIR} 2>&1 | tail -2
    os.chdir(REPO_DIR)

!pip install -q -e ".[bench]" sentencepiece tiktoken
print("Working directory:", os.getcwd())

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for myna-engine (pyproject.toml) ... done
Working directory: /content/myna-repo


In [5]:
# --- CELL 4: Multi-Question Document Bundle Loader ---
import json, hashlib

def load_bundles(path):
    bundles = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            state = d['state']
            src = d['_meta']['source']
            questions = []
            for qname, q in d['questions'].items():
                qtype = q['type']
                if qtype == 'choice':
                    opts = list(q['criteria'].keys())
                    gold = opts.index(q['label']) if q['label'] in opts else 0
                    questions.append({
                        'qname': qname, 'type': 'choice',
                        'instructions': q['instructions'],
                        'options': opts, 'descs': [q['criteria'][o] for o in opts],
                        'gold': gold
                    })
                elif qtype == 'noul':
                    gold = 1.0 if q['label'] else 0.0
                    questions.append({
                        'qname': qname, 'type': 'noul',
                        'instructions': q['instructions'],
                        'options': ['false', 'true'],
                        'descs': ['statement is false', 'statement is true'],
                        'gold': gold
                    })
                elif qtype == 'score':
                    levels = q['criteria']
                    gold = int(q['label']) if isinstance(q['label'], (int, float)) else 0
                    questions.append({
                        'qname': qname, 'type': 'score',
                        'instructions': q['instructions'],
                        'options': [str(i) for i in range(len(levels))],
                        'descs': levels,
                        'gold': gold
                    })
            if questions:
                bundles.append({'source': src, 'state': state, 'questions': questions})
    return bundles

train_bundles = load_bundles('data/decision-v2-pilot/train.jsonl')
dev_bundles = load_bundles('data/decision-v2-pilot/development.jsonl')
test_bundles = load_bundles('data/decision-v2-pilot/test.jsonl')

print(f"train bundles: {len(train_bundles)} | dev: {len(dev_bundles)} | test: {len(test_bundles)}")

train bundles: 57904 | dev: 1176 | test: 1176


In [6]:
# --- CELL 5: Load Tokenizer & Base Model ---
import torch
from transformers import AutoTokenizer, AutoModel

try:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
except Exception:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

base = AutoModel.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
print("base hidden size:", base.config.hidden_size)

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  874MB            

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  874MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


base hidden size: 1024


In [7]:
# --- CELL 6: Example Builder ---
MASK_ID = tok.mask_token_id
CLS_ID = tok.cls_token_id
SEP_ID = tok.sep_token_id
PAD_ID = tok.pad_token_id

def text_of(x):
    if isinstance(x, str): return x
    if x is None: return ''
    if isinstance(x, dict): return ' | '.join(f"{k}: {text_of(v)}" for k, v in x.items())
    if isinstance(x, list): return ' '.join(text_of(v) for v in x)
    return str(x)

def build_bundle_examples(bundle):
    state_text = text_of(bundle['state'])
    encoded_qs = []
    for q in bundle['questions']:
        n_opt = len(q['options'])
        max_opt_toks = max(2, (MAX_LEN - 96) // max(1, n_opt) - 2)

        opt_chunks = []
        for label, desc in zip(q['options'], q['descs']):
            opt_text = label.replace('_', ' ')
            d = text_of(desc)
            if d and max_opt_toks > 4:
                opt_text += ': ' + ' '.join(d.split()[:10])
            opt_ids = tok(opt_text, add_special_tokens=False)['input_ids'][:max_opt_toks]
            if not opt_ids: opt_ids = [tok.unk_token_id]
            opt_chunks.append(opt_ids + [MASK_ID, SEP_ID])

        prefix = f"Task ({bundle['source']} / {q['qname']}): {text_of(q['instructions'])} | Context: "
        prefix_ids = tok(prefix, add_special_tokens=False)['input_ids'][:64]
        state_ids = tok(state_text, add_special_tokens=False)['input_ids'][: (MAX_LEN - 64 - sum(len(c) for c in opt_chunks))]

        ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID]
        mask_pos = []
        for chunk in opt_chunks:
            if len(ids) + len(chunk) > MAX_LEN: break
            ids += chunk
            mask_pos.append(len(ids) - 2)

        encoded_qs.append({
            'qname': q['qname'],
            'type': q['type'],
            'input_ids': ids,
            'mask_pos': mask_pos,
            'gold': q['gold'],
            'n_opt': len(mask_pos)
        })
    return {'source': bundle['source'], 'questions': encoded_qs}

print("Sample built bundle test:", len(build_bundle_examples(train_bundles[0])['questions']))

Sample built bundle test: 3


In [10]:
# --- CELL 7: Learned Bilinear Question-Graph Head (Fixed Shapes) ---
import torch.nn as nn
import torch.nn.functional as F

class QuestionGraphHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, base_model, input_ids, attn, mask_pos, opt_mask):
        B, M, L = input_ids.shape
        # Flatten batch and question dimensions to process through DeBERTa safely
        flat_ids = input_ids.view(B * M, L)
        flat_attn = attn.view(B * M, L)

        out = base_model(input_ids=flat_ids, attention_mask=flat_attn).last_hidden_state

        _, _, O = mask_pos.shape
        flat_mpos = mask_pos.view(B * M, O)

        idx = torch.arange(B * M, device=input_ids.device).unsqueeze(1)
        h_options = out[idx, flat_mpos] # (B*M, O, hidden)

        logits = self.scorer(h_options).squeeze(-1).float()
        logits = logits.view(B, M, O)

        omask_flat = opt_mask.view(B, M, O)
        logits = logits.masked_fill(omask_flat == 0, -1e9)
        return logits

def bundle_collate(bundle_list):
    max_q = max(len(b['questions']) for b in bundle_list)
    max_len = max(max(len(q['input_ids']) for q in b['questions']) for b in bundle_list)
    max_opt = max(max(len(q['mask_pos']) for q in b['questions']) for b in bundle_list)
    B = len(bundle_list)

    ids = torch.full((B, max_q, max_len), PAD_ID, dtype=torch.long)
    attn = torch.zeros((B, max_q, max_len), dtype=torch.long)
    mpos = torch.zeros((B, max_q, max_opt), dtype=torch.long)
    omask = torch.zeros((B, max_q, max_opt), dtype=torch.long)
    golds = []

    for i, b in enumerate(bundle_list):
        g_list = []
        for j, q in enumerate(b['questions']):
            ids[i, j, :len(q['input_ids'])] = torch.tensor(q['input_ids'])
            attn[i, j, :len(q['input_ids'])] = 1
            mpos[i, j, :len(q['mask_pos'])] = torch.tensor(q['mask_pos'])
            omask[i, j, :len(q['mask_pos'])] = 1
            g_list.append(q['gold'])
        golds.append(g_list)
    return ids, attn, mpos, omask, golds, bundle_list

torch.manual_seed(SEED)
model = QuestionGraphHead(base.config.hidden_size)
print("Question Graph Head initialized successfully with fixed tensor flattening.")

Question Graph Head initialized successfully with fixed tensor flattening.


In [12]:
# --- CELL 8: SMOKE TEST (Fixed Cross Entropy Dtype) ---
from torch.utils.data import DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base.to(device)
model.to(device)

smoke_data = [build_bundle_examples(x) for x in train_bundles[:SMOKE_BUNDLES]]
smoke_dl = DataLoader(smoke_data, batch_size=2, shuffle=True, collate_fn=bundle_collate)

model.train()
opt = torch.optim.AdamW(model.parameters(), lr=HEAD_LR)

for ids, attn, mpos, omask, golds, _ in smoke_dl:
    ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)

    # Forward pass
    logits = model(base, ids, attn, mpos, omask) # (B, M, O)

    loss = 0.0
    for j in range(logits.size(1)):
        q_logits = logits[:, j, :].float() # Ensure float32 for cross_entropy
        # CRITICAL FIX: target must be torch.long, not float/int32
        q_golds = torch.tensor([g[j] for g in golds], dtype=torch.long, device=device)

        loss += F.cross_entropy(q_logits, q_golds)

    loss.backward()
    opt.step()
    opt.zero_grad()
    print("Smoke step loss successfully computed:", loss.item())
    break

print("SMOKE TEST PASSED.")

Smoke step loss successfully computed: 2.75095796585083
SMOKE TEST PASSED.


In [19]:
# --- CELL 9: RESUME AND RUN EPOCH 2 WITH GRADIENT CHECKPOINTING ---
import math, time, random
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

random.seed(SEED)
torch.manual_seed(SEED)

train_built = [build_bundle_examples(x) for x in train_bundles]
dev_built = [build_bundle_examples(x) for x in dev_bundles[:200]]
print(f"Training bundles built: {len(train_built)} | Dev bundles: {len(dev_built)}")

@torch.no_grad()
def evaluate_dev_bundles():
    model.eval()
    dl_dev = DataLoader(dev_built, batch_size=2, shuffle=False, collate_fn=bundle_collate)
    correct, total = 0, 0
    for ids, attn, mpos, omask, golds, raw_bundles in dl_dev:
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask)

        for b_idx, b in enumerate(raw_bundles):
            n_qs = len(b['questions'])
            for j in range(n_qs):
                pred = logits[b_idx, j, :].argmax(dim=-1).item()
                gold = golds[b_idx][j]
                correct += int(pred == gold)
                total += 1
    return correct / max(total, 1)

best_ckpt_path = f"{OUT_DIR}/myna-v5-best.pt"
best_dev_acc = 0.50  # From Epoch 1 completion

# Load the saved Epoch 1 checkpoint so we skip retraining it
if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state'])
    best_dev_acc = ckpt.get('dev_acc', 0.50)
    print(f"Successfully loaded best Epoch 1 checkpoint from {best_ckpt_path} (dev_acc={best_dev_acc:.4f})")
else:
    raise FileNotFoundError(f"Checkpoint not found at {best_ckpt_path}! Make sure Drive is mounted.")

def run_finetune_epoch(epoch):
    global best_dev_acc
    # Unfreeze base model and enable gradient checkpointing to save VRAM
    for p in base.parameters():
        p.requires_grad = True
    if hasattr(base, "gradient_checkpointing_enable"):
        base.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled on base model.")

    params = [
        {'params': base.parameters(), 'lr': BASE_LR},
        {'params': model.parameters(), 'lr': HEAD_LR * 0.5}
    ]

    opt = torch.optim.AdamW(params, weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda', enabled=(AMP_DTYPE == torch.float16))
    dl = DataLoader(train_built, batch_size=BATCH_BUNDLES, shuffle=True, collate_fn=bundle_collate)
    steps = math.ceil(len(dl) / ACCUM_STEPS)

    sched = get_cosine_schedule_with_warmup(opt, int(0.1 * steps), steps)

    model.train()
    opt.zero_grad()
    t0 = time.time()
    tot, n = 0.0, 0

    for i, (ids, attn, mpos, omask, golds, raw_bundles) in enumerate(dl):
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)

        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask)
            loss = 0.0
            for b_idx, b in enumerate(raw_bundles):
                n_qs = len(b['questions'])
                for j in range(n_qs):
                    q_logits = logits[b_idx, j, :].float()
                    q_gold = torch.tensor(golds[b_idx][j], dtype=torch.long, device=device)
                    loss += F.cross_entropy(q_logits.unsqueeze(0), q_gold.unsqueeze(0))
            loss = loss / ACCUM_STEPS

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"WARNING: NaN/Inf loss at epoch {epoch} batch {i+1}")
            opt.zero_grad()
            continue

        scaler.scale(loss).backward()
        tot += loss.item() * ACCUM_STEPS
        n += 1

        if (i + 1) % ACCUM_STEPS == 0 or i == len(dl) - 1:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(opt)
            scaler.update()

            sched.step()
            opt.zero_grad()

        if (i + 1) % 400 == 0:
            print(f"Epoch {epoch} (finetune) batch {i+1}/{len(dl)} loss {tot/max(n,1):.4f} elapsed {time.time()-t0:.0f}s")

    dev_acc = evaluate_dev_bundles()
    print(f"Epoch {epoch} fine-tune done | Mean train loss: {tot/max(n,1):.4f} | Dev Bundle Accuracy: {dev_acc:.4f}")

    if dev_acc >= best_dev_acc:
        best_dev_acc = dev_acc
        torch.save({'state': model.state_dict(), 'dev_acc': best_dev_acc}, best_ckpt_path)
        print(f"  -> New best dev accuracy saved to {best_ckpt_path}")

    gc.collect()
    torch.cuda.empty_cache()

# Run Epoch 2 safely with gradient checkpointing
run_finetune_epoch(2)
print("Training complete!")

Training bundles built: 57904 | Dev bundles: 200
Successfully loaded best Epoch 1 checkpoint from /content/drive/MyDrive/myna-v5-prototype/myna-v5-best.pt (dev_acc=0.5000)
Gradient checkpointing enabled on base model.
Epoch 2 (finetune) batch 400/57904 loss 1.2460 elapsed 76s
Epoch 2 (finetune) batch 800/57904 loss 1.2298 elapsed 152s
Epoch 2 (finetune) batch 1200/57904 loss 1.2532 elapsed 228s
Epoch 2 (finetune) batch 1600/57904 loss 1.2833 elapsed 304s
Epoch 2 (finetune) batch 2000/57904 loss 1.2878 elapsed 379s
Epoch 2 (finetune) batch 2400/57904 loss 1.2702 elapsed 455s
Epoch 2 (finetune) batch 2800/57904 loss 1.2629 elapsed 531s
Epoch 2 (finetune) batch 3200/57904 loss 1.2598 elapsed 608s
Epoch 2 (finetune) batch 3600/57904 loss 1.2413 elapsed 684s
Epoch 2 (finetune) batch 4000/57904 loss 1.2225 elapsed 761s
Epoch 2 (finetune) batch 4400/57904 loss 1.2086 elapsed 837s
Epoch 2 (finetune) batch 4800/57904 loss 1.1974 elapsed 913s
Epoch 2 (finetune) batch 5200/57904 loss 1.1856 elaps

In [20]:
# --- CELL 10: EVALUATION ON TEST BUNDLES ---
test_built = [build_bundle_examples(x) for x in test_bundles]

@torch.no_grad()
def evaluate_test():
    model.eval()
    dl_test = DataLoader(test_built, batch_size=4, shuffle=False, collate_fn=bundle_collate)
    correct, total = 0, 0
    primitive_stats = {'choice': [0, 0], 'score': [0, 0], 'noul': [0, 0]}

    for ids, attn, mpos, omask, golds, raw_bundles in dl_test:
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask)

        for b_idx, b in enumerate(raw_bundles):
            for j, q in enumerate(b['questions']):
                pred = logits[b_idx, j, :].argmax(dim=-1).item()
                gold = golds[b_idx][j]
                ptype = q['type']

                is_correct = int(pred == gold)
                correct += is_correct
                total += 1
                if ptype in primitive_stats:
                    primitive_stats[ptype][0] += is_correct
                    primitive_stats[ptype][1] += 1

    overall_acc = correct / max(total, 1)
    print(f"=== TEST EVALUATION REPORT ===")
    print(f"Overall Multi-Primitive Accuracy: {correct}/{total} = {overall_acc:.4f}")
    for ptype, (c, t) in primitive_stats.items():
        if t > 0:
            print(f"  - {ptype}: {c}/{t} = {c/t:.4f}")
    return overall_acc, primitive_stats

test_acc, stats = evaluate_test()

=== TEST EVALUATION REPORT ===
Overall Multi-Primitive Accuracy: 1162/1440 = 0.8069
  - choice: 541/696 = 0.7773
  - score: 152/240 = 0.6333
  - noul: 469/504 = 0.9306


In [21]:
# --- CELL 11: WITNESS JSON GENERATION ---
import datetime

witness = {
    'model_label': 'myna-v5-prototype (DeBERTa-v3-large + Question-Graph Multi-Primitive Head)',
    'base_model': BASE_MODEL,
    'seed': SEED,
    'best_dev_acc': best_dev_acc,
    'test_accuracy': test_acc,
    'timestamp': datetime.datetime.now().isoformat(),
    'notes': 'Evaluated across all 3 primitives (choice, score, noul) grouped by document bundles without hardcoded argmax hacks.'
}

witness_path = f"{OUT_DIR}/witness-v5.json"
with open(witness_path, 'w', encoding='utf-8') as f:
    json.dump(witness, f, indent=2)

print("Witness saved successfully to:", witness_path)
print(json.dumps(witness, indent=2))

Witness saved successfully to: /content/drive/MyDrive/myna-v5-prototype/witness-v5.json
{
  "model_label": "myna-v5-prototype (DeBERTa-v3-large + Question-Graph Multi-Primitive Head)",
  "base_model": "microsoft/deberta-v3-large",
  "seed": 42,
  "best_dev_acc": 0.65,
  "test_accuracy": 0.8069444444444445,
  "timestamp": "2026-10-31T22:22:59.832504",
  "notes": "Evaluated across all 3 primitives (choice, score, noul) grouped by document bundles without hardcoded argmax hacks."
}


In [23]:
# --- FIXED CELL 12: SAVE FULL BACKBONE + COMPUTE PAPER AUDIT ---
from collections import defaultdict

# 1. Save the fine-tuned DeBERTa-v3-large backbone using the active tokenizer reference
try:
    base.save_pretrained(f"{OUT_DIR}/deberta-v5-finetuned")
    if 'tokenizer' in globals():
        tokenizer.save_pretrained(f"{OUT_DIR}/deberta-v5-finetuned")
    elif 'tok' in globals():
        tok.save_pretrained(f"{OUT_DIR}/deberta-v5-finetuned")
    print("Saved fine-tuned DeBERTa backbone and tokenizer to Google Drive!")
except Exception as e:
    print(f"Note on save: {e}")

@torch.no_grad()
def full_paper_audit(bundles_raw, split_name="test"):
    model.eval()
    built = [build_bundle_examples(x) for x in bundles_raw]
    dl = DataLoader(built, batch_size=4, shuffle=False, collate_fn=bundle_collate)

    total_q, correct_q = 0, 0
    total_bundles, exact_bundles = 0, 0
    multi_q_bundles, contradicted_bundles = 0, 0
    cell_stats = defaultdict(lambda: [0, 0])
    prim_stats = defaultdict(lambda: [0, 0])

    for ids, attn, mpos, omask, golds, raw_b in dl:
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask)

        for b_idx, b in enumerate(raw_b):
            qs = b['questions']
            b_all_correct = True
            preds_by_type = {}

            for j, q in enumerate(qs):
                pred = logits[b_idx, j, :].argmax(dim=-1).item()
                gold = golds[b_idx][j]
                ok = int(pred == gold)

                correct_q += ok
                total_q += 1
                if not ok:
                    b_all_correct = False

                ptype = q['type']
                prim_stats[ptype][0] += ok
                prim_stats[ptype][1] += 1

                src = b.get('source', 'unknown')
                qid = q.get('qid', q.get('id', ptype))
                cell_stats[(src, qid)][0] += ok
                cell_stats[(src, qid)][1] += 1
                preds_by_type[ptype] = (pred, q)

            total_bundles += 1
            if b_all_correct:
                exact_bundles += 1

            if len(qs) > 1:
                multi_q_bundles += 1
                if 'choice' in preds_by_type and 'noul' in preds_by_type:
                    c_pred, c_q = preds_by_type['choice']
                    n_pred, n_q = preds_by_type['noul']
                    c_opts = list(c_q.get('criteria', {}).keys()) if isinstance(c_q.get('criteria'), dict) else []
                    if c_pred < len(c_opts):
                        chosen_label = str(c_opts[c_pred]).lower()
                        n_inst = str(n_q.get('instructions', '')).lower()
                        if chosen_label in n_inst and n_pred == 0:
                            contradicted_bundles += 1

    overall_acc = correct_q / max(total_q, 1)
    bundle_exact_acc = exact_bundles / max(total_bundles, 1)
    contradiction_rate = contradicted_bundles / max(multi_q_bundles, 1)

    per_cell_accs = {f"{s}/{q}": c / max(t, 1) for (s, q), (c, t) in sorted(cell_stats.items())}
    macro_acc = sum(per_cell_accs.values()) / max(len(per_cell_accs), 1)

    print(f"\n=== {split_name.upper()} FULL PAPER AUDIT ===")
    print(f"Overall Question Accuracy: {correct_q}/{total_q} = {overall_acc:.4f}")
    print(f"Unweighted Per-Cell Macro ({len(per_cell_accs)} cells): {macro_acc:.4f}")
    print(f"Joint Bundle Exact-Match Accuracy: {exact_bundles}/{total_bundles} = {bundle_exact_acc:.4f}")
    print(f"Multi-Question Contradiction Rate: {contradicted_bundles}/{multi_q_bundles} = {contradiction_rate:.4f}")

    return {
        "overall_acc": overall_acc,
        "macro_acc": macro_acc,
        "num_cells": len(per_cell_accs),
        "bundle_exact_acc": bundle_exact_acc,
        "contradiction_rate": contradiction_rate,
        "per_cell": per_cell_accs,
        "primitives": {k: c / max(t, 1) for k, (c, t) in prim_stats.items()}
    }

dev_audit = full_paper_audit(dev_bundles, "dev")
test_audit = full_paper_audit(test_bundles, "test")

witness['full_dev_audit'] = dev_audit
witness['full_test_audit'] = test_audit
with open(f"{OUT_DIR}/witness-v5-full.json", "w") as f:
    json.dump(witness, f, indent=2)
print("Saved complete paper witness to witness-v5-full.json!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned DeBERTa backbone and tokenizer to Google Drive!

=== DEV FULL PAPER AUDIT ===
Overall Question Accuracy: 1161/1440 = 0.8063
Unweighted Per-Cell Macro (14 cells): 0.8024
Joint Bundle Exact-Match Accuracy: 912/1176 = 0.7755
Multi-Question Contradiction Rate: 0/172 = 0.0000

=== TEST FULL PAPER AUDIT ===
Overall Question Accuracy: 1162/1440 = 0.8069
Unweighted Per-Cell Macro (14 cells): 0.7988
Joint Bundle Exact-Match Accuracy: 909/1176 = 0.7730
Multi-Question Contradiction Rate: 0/172 = 0.0000
Saved complete paper witness to witness-v5-full.json!
